# Host Regions With Unpredicted Coverage Enrichment

This exploratory report looks for enriched host-DNA segments outside geNomad predictions. Orange marks geNomad-predicted viral regions; red marks unpredicted enriched segments. Coverage is from the existing full-read mapping to each sample's assigned unmasked host. No reads are remapped, and no additional classifier or CheckV analysis is run.

These are coverage candidates, not new prophage predictions or activity calls. In preparations without DNase treatment before capsid lysis, free host DNA cannot be distinguished from protected/packaged DNA using enrichment alone. Repeats, cross-mapping, strain differences and copy-number differences can also produce peaks. This report does not establish transduction, gene expression or gene transfer, and never changes contig retention or isolate decisions.


In [ ]:
from pathlib import Path
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO
from IPython.display import display, Image, Markdown

sns.set_style("ticks", {"axes.grid": True})
sns.set_palette("colorblind")
plt.rcParams.update({"axes.linewidth": 1.5, "axes.labelsize": 14, "axes.titlesize": 14,
                     "xtick.labelsize": 10, "ytick.labelsize": 10, "svg.fonttype": "none",
                     "font.family": "sans-serif", "font.sans-serif": ["Liberation Sans", "DejaVu Sans"]})
regions = pd.read_csv(snakemake.input.regions, sep="\t", dtype={"host": str, "viral_id": str, "parent_contig": str})
assignments = pd.read_csv(snakemake.input.assignments, sep="\t", dtype={"sample": str, "expected_host": str}).set_index("sample")["expected_host"].to_dict()
host_lengths = {}
for host, path in zip(snakemake.params.hosts, snakemake.input.host_fastas):
    with open(path) as handle:
        host_lengths[host] = {record.id: len(record) for record in SeqIO.parse(handle, "fasta")}
figure_dir = Path(snakemake.output.figures)
figure_dir.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame([dict(bin_bp=snakemake.params.bin_bp, min_segment_length_bp=snakemake.params.min_length_bp,
    min_ratio=snakemake.params.min_ratio, min_local_ratio=snakemake.params.min_local_ratio,
    min_cohen_d=snakemake.params.min_cohen_d, min_mean_depth=snakemake.params.min_mean_depth,
    breadth_min_depth=snakemake.params.breadth_min_depth, min_breadth_percent=snakemake.params.min_breadth_percent,
    scaffold_end_mask_bp=snakemake.params.mask_bp, local_flank_bp=snakemake.params.local_flank_bp,
    plot_flank_bp=snakemake.params.plot_flank_bp)]))


## Coverage-First Region Screening

All assigned-host contigs are scanned, including those without geNomad predictions. Known embedded prophages and scaffold-end masks are excluded; whole-contig viral candidates are not scanned for additional host regions. Contig names remain text, including numeric and zero-padded names. Unreported positions are restored as zero depth.

Non-overlapping 1 kb bins must have mean depth >=5x, breadth >=80% at >=1x depth and enrichment >=2-fold relative to the unpredicted background on the same parent contig, by default. Minimum mean depth and the base depth used to calculate breadth are separate settings. Only full bins entirely outside the exclusions are eligible. Adjacent passing bins form a segment; segments must be at least 5 kb by default. The segment's ratio, Cohen's d (>=0.70), mean depth and breadth are then checked using unbinned per-base coverage, excluding both known prophages and the focal segment from the whole-parent-contig background. These discovery thresholds have independent `host_enriched_regions_*` settings; predicted-prophage activity settings are unchanged. Passing is labelled enrichment, not activity.

Each segment must also be >=2-fold enriched against its local flanks, using 10 kb on each side by default and excluding geNomad regions and scaffold-end masks. Flanks are clipped at contig boundaries. Zero or unavailable whole-contig or local background is not assessable, not infinite enrichment; these segments are not reported as hits. Local flank measurements remain in the table. Boundaries have the scan's bin resolution, not validated viral boundaries. Primary alignments can still represent non-unique placements.


In [ ]:
rows, sample_summaries = [], []
for sample, path in zip(snakemake.params.samples, snakemake.input.depth):
    host = assignments[sample]
    selected = regions[regions["host"].eq(host)]
    depths = {parent: np.zeros(length, dtype=float) for parent, length in host_lengths[host].items()}
    with gzip.open(path, "rt") as handle:
        has_coverage = bool(handle.read(1))
    if has_coverage:
        for chunk in pd.read_csv(path, sep="\t", names=["contig", "position", "depth"], dtype={"contig": str}, chunksize=100000):
            for parent, values in chunk[chunk["contig"].isin(depths)].groupby("contig"):
                depths[parent][values["position"].to_numpy(dtype=int)] = values["depth"].to_numpy(dtype=float)
    sample_start = len(rows)
    assessed_contigs, zero_background_contigs, viral_contigs, eligible_bases = 0, 0, 0, 0
    for parent, depth in depths.items():
        known = selected[selected["parent_contig"].eq(parent)]
        if known["evidence_scope"].eq("whole_contig_viral_candidate").any():
            viral_contigs += 1
            continue
        embedded = known[known["evidence_scope"].eq("embedded_prophage")]
        valid = np.ones(len(depth), dtype=bool)
        if snakemake.params.mask_bp:
            valid[:snakemake.params.mask_bp] = False
            valid[-snakemake.params.mask_bp:] = False
        for region in embedded.itertuples(index=False):
            start, end = sorted([int(region.start), int(region.end)])
            valid[start - 1:end] = False
        background = depth[valid]
        baseline = float(background.mean()) if background.size else np.nan
        if not baseline > 0:
            zero_background_contigs += 1
            continue
        n_bins = len(depth) // snakemake.params.bin_bp
        bins = depth[:n_bins * snakemake.params.bin_bp].reshape(n_bins, snakemake.params.bin_bp)
        eligible = valid[:n_bins * snakemake.params.bin_bp].reshape(n_bins, snakemake.params.bin_bp).all(axis=1)
        assessed_contigs += int(eligible.any())
        eligible_bases += int(eligible.sum()) * snakemake.params.bin_bp
        means = bins.mean(axis=1)
        breadths = 100 * np.count_nonzero(bins >= snakemake.params.breadth_min_depth, axis=1) / snakemake.params.bin_bp
        elevated = eligible & (means >= max(snakemake.params.min_mean_depth, snakemake.params.min_ratio * baseline)) & (breadths >= snakemake.params.min_breadth_percent)
        edges = np.diff(np.r_[False, elevated, False].astype(int))
        for left_bin, right_bin in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
            left, right = left_bin * snakemake.params.bin_bp, right_bin * snakemake.params.bin_bp
            if right - left < snakemake.params.min_length_bp:
                continue
            segment = depth[left:right]
            background_mask = valid.copy()
            background_mask[left:right] = False
            background = depth[background_mask]
            segment_mean = float(segment.mean())
            background_mean = float(background.mean()) if background.size else np.nan
            ratio = segment_mean / background_mean if background_mean > 0 else np.nan
            pooled_sd = np.sqrt((segment.var() + background.var()) / 2) if background.size else np.nan
            effect = (segment_mean - background_mean) / pooled_sd if pooled_sd > 0 else (
                np.inf if background_mean > 0 and segment_mean > background_mean else np.nan)
            breadth = 100 * np.count_nonzero(segment >= snakemake.params.breadth_min_depth) / segment.size
            if not (ratio >= snakemake.params.min_ratio and effect >= snakemake.params.min_cohen_d and
                    segment_mean >= snakemake.params.min_mean_depth and breadth >= snakemake.params.min_breadth_percent):
                continue
            flank_mask = np.zeros(len(depth), dtype=bool)
            flank_mask[max(0, left - snakemake.params.local_flank_bp):left] = True
            flank_mask[right:min(len(depth), right + snakemake.params.local_flank_bp)] = True
            flanks = depth[flank_mask & valid]
            flank_mean = float(flanks.mean()) if flanks.size else np.nan
            local_ratio = segment_mean / flank_mean if flank_mean > 0 else np.nan
            if not local_ratio >= snakemake.params.min_local_ratio:
                continue
            distances = [max(min(int(region.start), int(region.end)) - 1 - right,
                             left - max(int(region.start), int(region.end)), 0) for region in embedded.itertuples(index=False)]
            rows.append(dict(host=host, sample=sample, parent_contig=parent, start=left + 1, end=right,
                length_bp=right - left, host_contig_length_bp=len(depth), evidence_class="unpredicted_enriched_region",
                segment_mean_depth=segment_mean, background_mean_depth=background_mean,
                segment_background_ratio=ratio, cohen_d=effect, breadth_percent=breadth,
                covered_bases=int(np.count_nonzero(segment)), background_length_bp=int(background.size),
                background_breadth_percent=100 * np.count_nonzero(background >= snakemake.params.breadth_min_depth) / background.size,
                local_flank_mean_depth=flank_mean, local_flank_ratio=local_ratio,
                local_flank_breadth_percent=100 * np.count_nonzero(flanks >= snakemake.params.breadth_min_depth) / flanks.size if flanks.size else np.nan,
                nearest_genomad_distance_bp=min(distances) if distances else np.nan))
    sample_rows = rows[sample_start:]
    sample_summaries.append(dict(host=host, sample=sample, enriched_segments=len(sample_rows),
        enriched_length_bp=sum(row["length_bp"] for row in sample_rows),
        assessed_contigs=assessed_contigs, eligible_bases=eligible_bases,
        zero_background_contigs=zero_background_contigs, excluded_whole_viral_contigs=viral_contigs,
        assessment="enriched_regions_found" if sample_rows else ("no_enriched_regions" if assessed_contigs else "no_assessable_background")))
enriched = pd.DataFrame(rows, columns=["host", "sample", "parent_contig", "start", "end", "length_bp",
    "host_contig_length_bp", "evidence_class", "segment_mean_depth", "background_mean_depth",
    "segment_background_ratio", "cohen_d", "breadth_percent", "covered_bases", "background_length_bp",
    "background_breadth_percent", "local_flank_mean_depth", "local_flank_ratio",
    "local_flank_breadth_percent", "nearest_genomad_distance_bp"])
summary = pd.DataFrame(sample_summaries, columns=["host", "sample", "enriched_segments", "enriched_length_bp",
    "assessed_contigs", "eligible_bases", "zero_background_contigs", "excluded_whole_viral_contigs", "assessment"]).sort_values(["host", "sample"])
summary.to_csv(snakemake.output.summary, sep="\t", index=False)


## Enriched Regions And Sample Summary

Coordinates and lengths refer to each sample's individual enriched blocks (1-based inclusive). Similar blocks across samples are not counted as independent biological elements. Overlapping display windows are joined only for plotting; enriched blocks themselves are never joined across gaps. All assigned samples remain in the summary, including those with no enriched regions.


In [ ]:
windows = []
for (host, parent), group in enriched.groupby(["host", "parent_contig"], sort=True):
    for region in group.sort_values(["start", "end", "sample"]).itertuples(index=False):
        left = max(1, int(region.start) - snakemake.params.plot_flank_bp)
        right = min(host_lengths[host][parent], int(region.end) + snakemake.params.plot_flank_bp)
        if windows and windows[-1]["host"] == host and windows[-1]["parent_contig"] == parent and left <= windows[-1]["plot_end"]:
            windows[-1]["plot_end"] = max(windows[-1]["plot_end"], right)
        else:
            windows.append(dict(host=host, parent_contig=parent, plot_start=left, plot_end=right,
                window_id=f"{host}_enriched_window_{len(windows) + 1:05d}"))
enriched["window_id"] = ""
for window in windows:
    selected = enriched["host"].eq(window["host"]) & enriched["parent_contig"].eq(window["parent_contig"]) & enriched["start"].ge(window["plot_start"]) & enriched["end"].le(window["plot_end"])
    enriched.loc[selected, "window_id"] = window["window_id"]
enriched = enriched.sort_values(["host", "parent_contig", "start", "sample"])
enriched.to_csv(snakemake.output.table, sep="\t", index=False)
pd.DataFrame(windows, columns=["host", "parent_contig", "plot_start", "plot_end", "window_id"]).to_csv(
    figure_dir / "coverage_windows.tsv", sep="\t", index=False)
display(summary)
with pd.option_context("display.max_columns", None):
    display(enriched.reindex(columns=["host", "sample", "parent_contig", "start", "end", "length_bp",
        "segment_mean_depth", "background_mean_depth", "segment_background_ratio", "cohen_d",
        "breadth_percent", "local_flank_ratio", "nearest_genomad_distance_bp"]))


## Enriched Host-Region Coverage Profiles

Blue is read depth, orange is an existing geNomad prediction, and red is an unpredicted enriched block in the displayed sample. Each block has 200,000 bp of context on each side by default, clipped at contig boundaries. Overlapping context windows across samples are merged for consistent comparisons. Figures follow host, parent contig, genomic window and full sample-ID order; only samples with an enriched block in the window are plotted.

A thin red bracket labels each block's length and segment/background ratio. The dashed grey baseline excludes all embedded geNomad regions; exact focal-segment background means and local flank ratios are in the table. The 200 kb flanks are display context, not the statistical background. Curves are binned only for display, to at most 2,000 points per figure; detection statistics use unbinned depth. Where available, the geNomad legend includes the region's existing CheckV quality and completeness. There is no inferred CheckV score for the red blocks.


In [ ]:
profile_index = []
depth_paths = dict(zip(snakemake.params.samples, snakemake.input.depth))
for host in sorted(enriched["host"].unique(), key=str.casefold):
    display(Markdown(f"### Host {host}"))
    for parent in sorted(enriched.loc[enriched["host"].eq(host), "parent_contig"].unique()):
        parent_regions = regions[regions["host"].eq(host) & regions["parent_contig"].eq(parent)]
        selected = enriched[enriched["host"].eq(host) & enriched["parent_contig"].eq(parent)]
        # Cache only plotting windows, not whole host genomes across all samples.
        profiles = []
        for sample in sorted(selected["sample"].unique(), key=str.casefold):
            depth = np.zeros(host_lengths[host][parent], dtype=float)
            for chunk in pd.read_csv(depth_paths[sample], sep="\t", names=["contig", "position", "depth"], dtype={"contig": str}, chunksize=100000):
                values = chunk[chunk["contig"].eq(parent)]
                depth[values["position"].to_numpy(dtype=int)] = values["depth"].to_numpy(dtype=float)
            valid = np.ones(len(depth), dtype=bool)
            if snakemake.params.mask_bp:
                valid[:snakemake.params.mask_bp] = False
                valid[-snakemake.params.mask_bp:] = False
            for region in parent_regions[parent_regions["evidence_scope"].eq("embedded_prophage")].itertuples(index=False):
                start, end = sorted([int(region.start), int(region.end)])
                valid[start - 1:end] = False
            baseline = float(depth[valid].mean())
            for window in [item for item in windows if item["host"] == host and item["parent_contig"] == parent]:
                hits = selected[selected["sample"].eq(sample) & selected["window_id"].eq(window["window_id"])]
                if hits.empty:
                    continue
                left, right = window["plot_start"] - 1, window["plot_end"]
                edges = np.unique(np.linspace(left, right, min(2001, right - left + 1), dtype=int))
                profiles.append(dict(sample=sample, window=window, baseline=baseline, hits=hits,
                    positions=(edges[:-1] + edges[1:]) / 2,
                    depth=np.array([depth[a:b].mean() for a, b in zip(edges[:-1], edges[1:])])))
        current_window = None
        for profile in sorted(profiles, key=lambda item: (item["window"]["plot_start"], item["sample"].casefold())):
            window, sample, hits = profile["window"], profile["sample"], profile["hits"]
            if window["window_id"] != current_window:
                display(Markdown(f"#### Contig {parent}: {window['plot_start']:,}–{window['plot_end']:,} bp"))
                current_window = window["window_id"]
            fig, ax = plt.subplots(figsize=(12, 4))
            ax.plot(profile["positions"], profile["depth"], color="#0173b2", linewidth=1, label="Read depth")
            visible = parent_regions[parent_regions["start"].le(window["plot_end"]) & parent_regions["end"].ge(window["plot_start"])]
            for region in visible.itertuples(index=False):
                quality = getattr(region, "CheckV_checkv_quality", "not reported")
                quality = "not reported" if pd.isna(quality) else quality
                completeness = pd.to_numeric(getattr(region, "CheckV_completeness", np.nan), errors="coerce")
                completeness_text = f"{completeness:.1f}%" if pd.notna(completeness) else "not reported"
                ax.axvspan(int(region.start) - 1, int(region.end), color="#de8f05", alpha=0.2,
                    label=f"geNomad {int(region.start):,}–{int(region.end):,} bp (CheckV: {quality}; {completeness_text})")
            for index, region in enumerate(hits.itertuples(index=False)):
                ax.axvspan(region.start - 1, region.end, color="#d62728", alpha=0.22,
                    label="Unpredicted enriched region" if index == 0 else None)
                level = 0.98 - 0.08 * (index % 3)
                ax.plot([region.start - 1, region.end], [level, level], color="#d62728", linewidth=1, transform=ax.get_xaxis_transform())
                ax.text((region.start - 1 + region.end) / 2, level - 0.01,
                    f"{region.length_bp:,} bp; {region.segment_background_ratio:.1f}×",
                    transform=ax.get_xaxis_transform(), color="#b22222", ha="center", va="top", fontsize=8)
            ax.axhline(profile["baseline"], color="grey", linestyle="--", linewidth=1, label="Unpredicted host-background mean")
            ax.set_xlim(window["plot_start"] - 1, window["plot_end"])
            ax.set_ylim(bottom=0)
            ax.set_title(f"Host {host} | {sample} | contig {parent}\n"
                f"Host contig length: {host_lengths[host][parent]:,} bp | {len(hits)} enriched block(s); {int(hits['length_bp'].sum()):,} bp total\n"
                "Orange: geNomad | red: unpredicted enrichment (not confirmed viral or packaged DNA)", fontsize=10)
            ax.set_xlabel("Position on host contig (bp)")
            ax.set_ylabel("Read depth")
            ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
            sns.despine(ax=ax)
            fig.tight_layout()
            png = figure_dir / (window["window_id"] + "__" + sample + ".png")
            svg = png.with_suffix(".svg")
            fig.savefig(png, dpi=300, bbox_inches="tight")
            fig.savefig(svg, bbox_inches="tight")
            plt.close(fig)
            profile_index.append(dict(host=host, sample=sample, parent_contig=parent, window_id=window["window_id"],
                plot_start=window["plot_start"], plot_end=window["plot_end"], enriched_blocks=len(hits),
                genomad_regions=len(visible), background_mean_depth=profile["baseline"], png=str(png), svg=str(svg)))
            display(Image(filename=str(png)))
for host in sorted(set(assignments[sample] for sample in snakemake.params.samples) - set(enriched["host"]), key=str.casefold):
    display(Markdown(f"### Host {host}\nNo unpredicted enriched regions passed the screening criteria. See the sample summary for background assessability."))
pd.DataFrame(profile_index, columns=["host", "sample", "parent_contig", "window_id", "plot_start", "plot_end",
    "enriched_blocks", "genomad_regions", "background_mean_depth", "png", "svg"]).to_csv(
    figure_dir / "coverage_profiles.tsv", sep="\t", index=False)
